# 00 BIDMC Dataset Audit

Audit raw BIDMC file coverage before preprocessing.

Inputs from: raw BIDMC files in `BIDMC_DATASET_DIR`.
Outputs to: `outputs/bidmc_dataset_file_coverage.csv`.
Next: notebook 01 for SpO2 label-readiness checks.

## Environment And Dataset Resolution

Resolve project paths, validate the BIDMC dataset directory, and define required file kinds for each recording.

In [ ]:
from pathlib import Path
import os
import pandas as pd

REQUIRED_FILE_KINDS = ('Signals', 'Numerics', 'Breaths', 'Fix')

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').is_dir() and (candidate / 'outputs').is_dir():
            return candidate
    raise RuntimeError('Could not find project root containing notebooks/ and outputs/')

def resolve_bidmc_dir(project_root: Path) -> Path:
    env_path = os.getenv('BIDMC_DATASET_DIR', '').strip()
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.is_dir():
            return p

    raise FileNotFoundError(
        'BIDMC dataset directory not found. Set BIDMC_DATASET_DIR in .env or environment.'
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_DIR = resolve_bidmc_dir(PROJECT_ROOT)
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
COVERAGE_CSV = OUTPUTS_DIR / 'bidmc_dataset_file_coverage.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir:  {DATASET_DIR}')


## Recording Coverage Summary

Enumerate recordings, verify all required file families are present, and write a per-recording coverage table for downstream notebooks.

In [ ]:
def infer_recording_id(path: Path):
    parts = path.stem.split('_')
    if len(parts) < 3 or parts[0] != 'bidmc':
        return None
    return '_'.join(parts[:2])

record_ids = sorted({rid for f in DATASET_DIR.glob('bidmc_*_*.*') for rid in [infer_recording_id(f)] if rid is not None})
rows = []
for recording_id in record_ids:
    row = {'recording_id': recording_id}
    for kind in REQUIRED_FILE_KINDS:
        csv_path = DATASET_DIR / f'{recording_id}_{kind}.csv'
        txt_path = DATASET_DIR / f'{recording_id}_{kind}.txt'
        selected = csv_path if csv_path.is_file() else (txt_path if txt_path.is_file() else None)
        row[f'has_{kind.lower()}'] = selected is not None
        row[f'{kind.lower()}_path'] = str(selected) if selected else ''
    row['all_required_present'] = all(row[f'has_{k.lower()}'] for k in REQUIRED_FILE_KINDS)
    rows.append(row)

coverage_df = pd.DataFrame(rows).sort_values('recording_id').reset_index(drop=True)
coverage_df.to_csv(COVERAGE_CSV, index=False)

print(f'Recordings discovered: {len(coverage_df)}')
print(f'Recordings with all required files: {int(coverage_df['all_required_present'].sum())}')
print(f'Wrote: {COVERAGE_CSV}')
coverage_df.head()
